In [ ]:
# Installing the brain (PyTorch), the model library (timm), and the UI tool (Gradio)
!pip install timm gradio torch torchvision matplotlib opencv-python ttach

In [ ]:
import torch
import timm
from PIL import Image
import torchvision.transforms as T
import numpy as np
import cv2
import gradio as gr

# 1. Load a pre-trained "Swin Transformer" - the modern choice for 2026
model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=5)
model.eval()

# 2. Preprocessing: This cleans the eye image so the AI can see clearly
def preprocess_image(img):
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return transform(img).unsqueeze(0)

# 3. Prediction Function
def predict(input_img):
    input_tensor = preprocess_image(input_img)
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)

    # Levels of Diabetic Retinopathy
    categories = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
    results = {categories[i]: float(probabilities[i]) for i in range(5)}

    return results

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [ ]:
# Create the Web UI
interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=3),
    title="Next-Gen Diabetic Retinopathy Detector",
    description="Upload a retinal fundus image. The Swin Transformer AI will analyze severity levels."
)

# This generates a PUBLIC link you can share!
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://83f590574544937518.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
import json

# 1. Clear old keys
!rm -rf ~/.kaggle
!mkdir -p ~/.kaggle

# 2. Use the NEWEST key you just made
user_data = {
    "username": "aaftab001",
    "key": "KGAT_496d6d008836d639c4ecd580f92f4354"
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(user_data, f)

!chmod 600 /root/.kaggle/kaggle.json

# 3. Try to list - if this fails, we must use the Manual Upload method above
!kaggle competitions list

In [ ]:
!pip install opendatasets --upgrade --quiet
import opendatasets as od

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from torchvision import transforms

# Ye wahi transform hai jo aapke interface mein 'preprocess_image' ke naam se hai
preprocess_image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print("✅ Transform ready!")

✅ Transform ready!


In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 1. Device Setup (Fixing the Error)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Simple Dataset Class
class DRDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        present_imgs = [f.split('.')[0] for f in os.listdir(img_dir) if f.endswith('.png')]
        self.df = self.df[self.df['id_code'].isin(present_imgs)].reset_index(drop=True)

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.df.iloc[idx, 0] + '.png')
        image = Image.open(img_path).convert('RGB')
        label = int(self.df.iloc[idx, 1])
        return self.transform(image), label

# 3. Model ko GPU par bhejna (Ye line error fix karegi)
model.to(device)

# 4. Data Loaders
train_ds = DRDataset('/content/drive/MyDrive/D.R project/train.csv',
                     '/content/drive/MyDrive/D.R project/train_images/',
                     preprocess_image_transform)

loader = DataLoader(train_ds, batch_size=16, shuffle=True)

# 5. Optimizer & Loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)
criterion = torch.nn.CrossEntropyLoss()

print(f"🔄 Training Started... Total Images: {len(train_ds)}")

# 6. Normal Training Loop
for epoch in range(15): # Epoch badha diye hain
    running_loss = 0.0
    correct = 0
    total = 0

    model.train()
    for imgs, labels in loader:
        # Dono ko GPU par bhejna
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Accuracy calculate karna
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Har epoch ke baad simple print
    epoch_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/15 - Loss: {running_loss/len(loader):.4f} - Accuracy: {epoch_acc:.2f}%")

    # Model save karna taaki mehnat waste na ho
    torch.save(model.state_dict(), '/content/drive/MyDrive/D.R project/swin_dr_final.pth')

print("🎉 Training Finished!")

🔄 Training Started... Total Images: 1674
Epoch 1/15 - Loss: 1.0114 - Accuracy: 69.77%
Epoch 2/15 - Loss: 0.9403 - Accuracy: 70.37%
Epoch 3/15 - Loss: 0.8236 - Accuracy: 72.94%
Epoch 4/15 - Loss: 0.6985 - Accuracy: 78.91%
Epoch 5/15 - Loss: 0.6559 - Accuracy: 80.05%
Epoch 6/15 - Loss: 0.6384 - Accuracy: 80.47%
Epoch 7/15 - Loss: 0.6225 - Accuracy: 80.65%
Epoch 8/15 - Loss: 0.6133 - Accuracy: 80.88%
Epoch 9/15 - Loss: 0.6060 - Accuracy: 80.41%
Epoch 10/15 - Loss: 0.5931 - Accuracy: 80.47%
Epoch 11/15 - Loss: 0.5916 - Accuracy: 80.47%
Epoch 12/15 - Loss: 0.5735 - Accuracy: 81.24%
Epoch 13/15 - Loss: 0.5631 - Accuracy: 81.90%
Epoch 14/15 - Loss: 0.5576 - Accuracy: 81.78%
Epoch 15/15 - Loss: 0.5591 - Accuracy: 81.54%
🎉 Training Finished!


In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# 1. Path Setup (Aapke folder names ke hisaab se)
base_path = '/content/drive/MyDrive/D.R project/'
csv_path = os.path.join(base_path, 'train.csv')
img_dir = os.path.join(base_path, 'train_images/')

# 2. Dataset Class: Ye aapke data aur model ko jodeyga
class DRDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        # Sirf wahi images uthayega jo folder mein maujood hain
        present_imgs = [f.split('.')[0] for f in os.listdir(img_dir) if f.endswith('.png')]
        self.df = self.df[self.df['id_code'].isin(present_imgs)].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.df.iloc[idx, 0] + '.png')
        image = Image.open(img_path).convert('RGB')
        label = int(self.df.iloc[idx, 1])
        return self.transform(image), label

# 3. Training Process
# Hum wahi 'model' use kar rahe hain jo aapne screenshot mein banaya tha
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_ds = DRDataset(csv_path, img_dir, preprocess_image_transform)
loader = DataLoader(train_ds, batch_size=2, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = torch.nn.CrossEntropyLoss()

print(f"🔄 Connection Established! Training starting on {len(train_ds)} images...")

model.train() # Model ko sikhne ke liye ready karein
for epoch in range(5):
    running_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"✅ Epoch {epoch+1}/5 - Loss: {running_loss/len(loader):.4f}")

print("\n🎉 Training Complete! Ab aapka model in images ko pehchanta hai.")
model.eval() # Training ke baad prediction mode mein wapas layein

🔄 Connection Established! Training starting on 1358 images...
✅ Epoch 1/5 - Loss: 0.6715
✅ Epoch 2/5 - Loss: 0.6046
✅ Epoch 3/5 - Loss: 0.5613
✅ Epoch 4/5 - Loss: 0.4711
✅ Epoch 5/5 - Loss: 0.6398

🎉 Training Complete! Ab aapka model in images ko pehchanta hai.


SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (layers): Sequential(
    (0): SwinTransformerStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): SwinTransformerBlock(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (drop_path1): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU(approximate='none')
            (drop1): 

In [ ]:
# Model ke weights ko Drive mein save karna
save_path = '/content/drive/MyDrive/D.R project/swin_dr_final.pth'
torch.save(model.state_dict(), save_path)
print(f"✅ Training saved! Ab aapko baar-baar train karne ki zaroorat nahi hai.")

✅ Training saved! Ab aapko baar-baar train karne ki zaroorat nahi hai.


In [ ]:
model.eval()

SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (layers): Sequential(
    (0): SwinTransformerStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): SwinTransformerBlock(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (drop_path1): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU(approximate='none')
            (drop1): 

In [ ]:
import torch
import torch.nn.functional as F

def predict(input_img):
    try:
        # 1. Model ko prediction mode mein rakhna
        model.eval()

        # 2. Image Preprocessing (Wahi jo humne pehle define kiya tha)
        # Ensure 'preprocess_image_transform' is already defined above!
        input_tensor = preprocess_image_transform(input_img).unsqueeze(0)

        # 3. Image ko model ke device par bhejna (CPU/GPU)
        input_tensor = input_tensor.to(device)

        # 4. Inference (Prediction)
        with torch.no_grad():
            output = model(input_tensor)
            # Swin Transformer tiny sometimes outputs a list or dict, let's handle it
            if isinstance(output, (list, tuple)):
                output = output[0]

            # 5. Probabilities calculate karna
            probabilities = F.softmax(output[0], dim=0)

        # 6. Diagnosis Categories
        categories = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']

        # Result ko dictionary mein convert karna
        results = {categories[i]: float(probabilities[i]) for i in range(5)}
        return results

    except Exception as e:
        # Agar koi error aaye toh wo interface mein dikhega
        return {"Error": str(e)}

# Interface ko phir se launch karein
interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=3),
    title="Next-Gen Diabetic Retinopathy Detector",
    description="Analyze retinal images for Diabetic Retinopathy stages using Swin Transformer."
)
interface.launch(share=True, inline=False)

NameError: name 'gr' is not defined

In [ ]:
import torch
import timm
import gradio as gr
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import os

# 1. Device & Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = timm.create_model('swin_tiny_patch4_window7_224', pretrained=False, num_classes=5)
model.to(device)

# 2. Path Fix: Aapne is naam se save kiya tha
model_path = '/content/drive/MyDrive/D.R project/swin_dr_final.pth'

if os.path.exists(model_path):
    # Load the weights
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint)
    model.eval()
    print("🚀 Balle Balle! Model successfully loaded from 'swin_dr_final.pth'")
else:
    print(f"❌ Abhi bhi nahi mila! Drive folder check karein. Expected path: {model_path}")

# 3. Preprocessing
preprocess_image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 4. Predict Function
def predict(input_img):
    try:
        input_tensor = preprocess_image_transform(input_img).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(input_tensor)
            if isinstance(output, (list, tuple)): output = output[0]
            probabilities = F.softmax(output[0], dim=0)

        categories = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
        return {categories[i]: float(probabilities[i]) for i in range(5)}
    except Exception as e:
        return {"Error": str(e)}

# 5. Interface
interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=3),
    title="Diabetic Retinopathy Detector (Swin Transformer)",
    description="Upload a retinal fundus image to detect severity levels."
)

interface.launch(share=True)

🚀 Balle Balle! Model successfully loaded from 'swin_dr_final.pth'


AttributeError: module 'gradio' has no attribute 'Request'

In [ ]:
!pip uninstall -y gradio
!pip install gradio==4.21.0  # Ek stable version install karte hain

Found existing installation: gradio 5.50.0
Uninstalling gradio-5.50.0:
  Successfully uninstalled gradio-5.50.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.8 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
  Attempting uninstall: tomlkit
    Found existing installation: tomlkit 0.13.3
    Uninstalling tomlkit-0.13.3:
      Successfully uninstalled tomlkit-0.13.3
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    